# ZemaVoice Stage 5 — Cinematic Narrator
Interactive Amharic narration: speed, sentence pauses, measured duration, and WAV download.

**Evaluation only:** Uses Meta `facebook/mms-tts-amh` under CC BY-NC 4.0. Not for commercial deployment. Presets change timing, not the model's learned emotion or speaker identity.

In [ ]:
!pip -q install 'transformers>=4.44,<5' 'accelerate>=0.26' 'soundfile>=0.12' 'uroman>=1.3' 'ipywidgets>=8'

In [ ]:
import torch, numpy as np, soundfile as sf, re, inspect
from IPython.display import Audio, display
from transformers import VitsModel, VitsTokenizer, set_seed
from uroman import Uroman
model_id='facebook/mms-tts-amh'
tokenizer=VitsTokenizer.from_pretrained(model_id)
model=VitsModel.from_pretrained(model_id)
romanizer=Uroman()
device='cuda' if torch.cuda.is_available() else 'cpu'
model=model.to(device).eval()
print('Ready:',device,'| Sample rate:',model.config.sampling_rate)
print('Forward parameters:',list(inspect.signature(model.forward).parameters))


In [ ]:
from IPython.display import display
import ipywidgets as widgets
presets={'Natural Conversation':(1.0,0.22),'Cinematic Narrator':(0.88,0.48),'Emotional Storytelling':(0.82,0.72)}
style=widgets.Dropdown(options=list(presets),value='Cinematic Narrator',description='Style:')
pace=widgets.FloatSlider(value=0.88,min=0.7,max=1.15,step=0.01,description='Speed:',readout_format='.2f')
pause=widgets.IntSlider(value=480,min=0,max=1200,step=40,description='Pause ms:')
text_input=widgets.Textarea(value='በአንድ ጨለማ ሌሊት፣ መንደሩ በጸጥታ ተውጦ ነበር። በዚያን ጊዜ ያልተጠበቀ ነገር ተፈጠረ።',description='Amharic:',layout=widgets.Layout(width='95%',height='120px'))
def preset_changed(change):
    if change['name']=='value':
        rate, gap=presets[change['new']];pace.value=rate;pause.value=int(gap*1000)
style.observe(preset_changed,names='value')
display(style,pace,pause,text_input)
print('Lower speaking_rate generally lengthens synthesized speech. Presets are pacing approximations, not emotional voice models.')


In [ ]:
import inspect
set_seed(42)
raw=text_input.value.strip()
assert raw, 'Please enter Amharic text.'
segments=[x.strip() for x in re.findall(r'[^።!?\n]+[።!?]?',raw) if x.strip()]
assert segments, 'No speech segments found.'
assert len(segments)<=30, 'Please use 30 sentences or fewer per generation.'
assert len(raw)<=3000, 'Please use 3000 characters or fewer.'
sr=model.config.sampling_rate
parts=[]
for i,sentence in enumerate(segments):
    romanized=romanizer.romanize_string(sentence)
    assert romanized, f'Romanization failed on segment {i+1}'
    inputs=tokenizer(romanized,return_tensors='pt').to(device)
    kwargs={}
    if 'speaking_rate' in inspect.signature(model.forward).parameters:
        kwargs['speaking_rate']=pace.value
    else:
        print('Warning: this Transformers model does not expose speaking_rate; speed slider may not affect synthesis.')
    with torch.no_grad():
        audio=model(**inputs,**kwargs).waveform[0].detach().cpu().float().numpy()
    parts.append(audio)
    print(f'Sentence {i+1}: {len(audio)/sr:.2f} sec')
    if i<len(segments)-1 and pause.value>0:
        parts.append(np.zeros(round(sr*pause.value/1000),dtype=np.float32))
waveform=np.concatenate(parts)
output='/content/zemavoice_cinematic_narration.wav'
sf.write(output,waveform,sr)
print(f'Total duration: {len(waveform)/sr:.2f} seconds | Preset: {style.value} | Rate: {pace.value:.2f} | Pause: {pause.value} ms')
display(Audio(waveform,rate=sr))


In [ ]:
from google.colab import files
files.download('/content/zemavoice_cinematic_narration.wav')
